# SLEAP Pipeline — Local (Modelo em Pasta Local)
**Moita Lab · Champalimaud Foundation**
Rodrigo Garrido

---
## CRITICAL WARNING — NVIDIA GPU

This pipeline runs deep neural networks. Performance depends heavily on your hardware:

| Situation | Estimated time per video |
|---|---|
| **NVIDIA GPU** (e.g., RTX 2060 or higher) | **2–10 minutes** |
| **CPU only** (no GPU) | **1–4 hours** |

> **Without a GPU, the pipeline will still run — but it is very slow.**  
> To check if your PC has a compatible GPU, run the validation cell (Step 2) before starting.

**Minimum requirements with GPU:**
- NVIDIA GPU with at least 4 GB VRAM
- Updated NVIDIA drivers (version ≥ 450)
- CUDA Toolkit 11.3 (installed automatically by `setup_env.bat`)

---

## Description

Local pipeline for *Drosophila* **pose estimation** powered by SLEAP.ai.

Combines two tracking systems:
- **Bonsai** — fly centroid (normalized 0–1 coordinates)
- **SLEAP** — 8 keypoints relative to the centroid (Head, Thorax, Abdomen, Left, Right, LeftWing, RightWing, Top)

Output: one CSV per fly containing all normalized positions (0–1) relative to the arena.

**References:**  
SLEAP.ai — https://doi.org/10.1038/s41592-022-01426-1  
Moita Lab — https://moitalab.org/

## Setup (once per PC)

Before using this notebook for the first time, run `setup_env.bat`  
(located in the same folder as this file).

The script automatically installs:
1. The `sleap_env` conda environment with SLEAP, TensorFlow, and CUDA
2. The `Python (sleap_env)` kernel in Jupyter

> **This notebook DOES NOT download the model from GitHub.**  
> Place the trained model folder (containing `best_model.h5` + `training_config.json`) in any local directory and point `MODEL_PATH` to that folder in Step 1.

> **Make sure you have this notebook open using the `Python (sleap_env)` kernel**  
> (top right corner in JupyterLab / VS Code)

## Execution Order

1. **Step 1** — Fill in the paths in the configuration cell
2. **Step 2** — Run the validation (checks folders and GPU)
3. **Step 3** — Run the pipeline (SLEAP inference + post-processing)

---
## Step 1 — Configuration

Edit the paths below before running any cell.

In [ ]:
import os

# ================================================================
# CONFIGURATION — edit these paths before running the pipeline
# ================================================================

# >>> DEFINE DIRECTORY HERE: root folder of your experiment <<<
EXPERIMENT_ROOT = r"E:\Champalimaud_Project_MoitaLab\Colab"

# >>> DEFINE DIRECTORY HERE: trained SLEAP model folder (8 keypoints) <<<
# Local model — NOT downloaded from GitHub in this notebook.
# Point directly to the folder containing best_model.h5 + training_config.json.
MODEL_PATH = os.path.join(EXPERIMENT_ROOT, "Sleap_Model")

# ----------------------------------------------------------------
# Subfolder structure (change only if your structure is different)
# ----------------------------------------------------------------
VIDEO_FOLDER   = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "CropRaw")
ARENAS_FOLDER  = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "Arenas")
TRACKED_FOLDER = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "Tracked")
OUTPUT_FOLDER  = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "Pose")
TEMP_FOLDER    = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "Temp_SLP")

print("Configuration loaded.")
print(f"  Experiment : {EXPERIMENT_ROOT}")
print(f"  Model      : {MODEL_PATH}")

---
## Step 2 — Validation

Checks that all folders exist and detects the available GPU.

In [ ]:
import tensorflow as tf

print("=" * 55)
print(" PATH VALIDATION")
print("=" * 55)

errors = []
checks = [
    ("Videos (CropRaw)",   VIDEO_FOLDER),
    ("Arenas",             ARENAS_FOLDER),
    ("Tracked (Bonsai)",   TRACKED_FOLDER),
    ("SLEAP Model",        MODEL_PATH),
]

for label, path in checks:
    if os.path.exists(path):
        n = len(os.listdir(path))
        print(f"  OK   {label}: {path} ({n} items)")
    else:
        errors.append(f"  ERROR {label}: {path}")
        print(f"  ERROR {label}: {path} -- NOT FOUND")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(TEMP_FOLDER,   exist_ok=True)
print(f"  OK   Output (Pose)   : {OUTPUT_FOLDER}")
print(f"  OK   Temp (.slp)     : {TEMP_FOLDER}")

print()
print("=" * 55)
print(" GPU CHECK")
print("=" * 55)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"  GPU detected: {len(gpus)} device(s)")
    for g in gpus:
        print(f"    - {g.name}")
    print("  Pipeline will run in GPU mode (fast).")
else:
    print("  WARNING: No NVIDIA GPU detected.")
    print("  The pipeline will run on CPU -- this may take hours per video.")

print()
if errors:
    print("PATH ERRORS FOUND. Fix them before continuing.")
    raise RuntimeError("Invalid paths — see messages above.")
else:
    print("All checks passed. You can run Step 3.")

---
## Step 3 — Pipeline

Run this cell to process all videos in the `CropRaw` folder.  
Already processed videos (existing `.slp`) will be automatically skipped.

In [ ]:
import os, sys, glob, subprocess
import sleap
import pandas as pd
import numpy as np
from PIL import Image

node_map = {
    'L': 'Left', 'R': 'Right', 'H': 'Head', 'Trx': 'Thorax',
    'Abd': 'Abdomen', 'Lw': 'LeftWing', 'Rw': 'RightWing', 'T': 'Top'
}

# ------------------------------------------------------------------
# 1. SLEAP Inference
# ------------------------------------------------------------------
print("=" * 55)
print(" SLEAP INFERENCE")
print("=" * 55)

video_files = sorted(glob.glob(os.path.join(VIDEO_FOLDER, "*.avi")) +
                     glob.glob(os.path.join(VIDEO_FOLDER, "*.mp4")))
if not video_files:
    raise RuntimeError(f"No video (.avi/.mp4) files found in: {VIDEO_FOLDER}")

print(f"  {len(video_files)} video(s) found.\n")

for video in video_files:
    c_id    = os.path.splitext(os.path.basename(video))[0]
    slp_out = os.path.join(TEMP_FOLDER, f"{c_id}.predictions.slp")

    if os.path.exists(slp_out):
        print(f"  [skip]  {c_id}  (already processed)")
        continue

    print(f"  [track] {c_id} ...")
    result = subprocess.run(
        ["sleap-track", video, "--model", MODEL_PATH,
         "-o", slp_out, "--no-empty-frames"],
        capture_output=False
    )
    if result.returncode != 0:
        print(f"  [ERROR] sleap-track failed for {c_id}")

# ------------------------------------------------------------------
# 2. Post-processing: SLEAP + Bonsai
# ------------------------------------------------------------------
print()
print("=" * 55)
print(" POST-PROCESSING: SLEAP + BONSAI")
print("=" * 55)

slp_files = sorted(glob.glob(os.path.join(TEMP_FOLDER, "*.predictions.slp")))
if not slp_files:
    raise RuntimeError(f"No .slp files found in: {TEMP_FOLDER}")

for slp_path in slp_files:
    c_id = os.path.basename(slp_path).replace(".predictions.slp", "")

    # Bonsai CSV name: replace _crop with _tracked
    tracked_id = c_id.replace("_crop", "_tracked")
    csv_path = os.path.join(TRACKED_FOLDER, f"{tracked_id}.csv")
    if not os.path.exists(csv_path):
        print(f"  [skip] Missing Bonsai CSV for {c_id}  (expected: {tracked_id}.csv)")
        continue

    print(f"  Processing: {c_id}")

    session_prefix = c_id.split('-fly')[0].split('_fly')[0]
    arena_candidates = [
        os.path.join(ARENAS_FOLDER, f"{c_id.replace('_crop', '')}.png"),
        os.path.join(ARENAS_FOLDER, f"{session_prefix}.png"),
    ]
    arena_img = next((p for p in arena_candidates if os.path.exists(p)), None)

    if arena_img:
        with Image.open(arena_img) as img:
            w_arena, h_arena = float(img.size[0]), float(img.size[1])
    else:
        tried = [os.path.basename(p) for p in arena_candidates]
        print(f"    Arena image not found (tried: {tried}) -- fallback to 1280x1024.")
        w_arena, h_arena = 1280.0, 1024.0

    df_t   = pd.read_csv(csv_path)
    labels = sleap.load_file(slp_path)

    col_x = [c for c in df_t.columns if 'X' in c.upper() and ('CENTROID' in c.upper() or len(c) == 1)][0]
    col_y = [c for c in df_t.columns if 'Y' in c.upper() and ('CENTROID' in c.upper() or len(c) == 1)][0]

    data = []
    for frame in labels:
        f_idx = frame.frame_idx
        c_row = df_t[df_t['FrameIndex'] == f_idx]
        if c_row.empty:
            continue

        cx_px = c_row[col_x].values[0] * w_arena
        cy_px = c_row[col_y].values[0] * h_arena
        row   = {'FrameIndex': f_idx}

        if len(frame.instances) == 0:
            for name in node_map.values():
                row[f'{name}.Position.X'] = np.nan
                row[f'{name}.Position.Y'] = np.nan
                row[f'{name}.Confidence'] = 0.0
        else:
            for inst in frame.instances:
                for node in labels.skeleton.nodes:
                    pt   = inst[node.name]
                    name = node_map.get(node.name, node.name)
                    if pt is not None:
                        norm_x = (cx_px + (pt.x - 64.0)) / w_arena
                        norm_y = (cy_px + (pt.y - 64.0)) / h_arena
                        row[f'{name}.Position.X'] = max(0.0, min(1.0, norm_x))
                        row[f'{name}.Position.Y'] = max(0.0, min(1.0, norm_y))
                        row[f'{name}.Confidence'] = pt.score
                    else:
                        row[f'{name}.Position.X'] = np.nan
                        row[f'{name}.Position.Y'] = np.nan
                        row[f'{name}.Confidence'] = 0.0
        data.append(row)

    if data:
        out_csv = os.path.join(OUTPUT_FOLDER, f"{c_id}_pose.csv")
        pd.DataFrame(data).sort_values('FrameIndex').to_csv(out_csv, index=False, na_rep='NaN')
        print(f"    Saved: {os.path.basename(out_csv)}")
    else:
        print(f"    No data found for {c_id}.")

print()
print("Pipeline finished successfully!")